In [1]:
import pandas as pd 
import numpy as np
from datetime import datetime, date

In [2]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
    version="v2",
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

https://api.allenneuraldynamics.org/v2/metadata_index/data_assets


In [3]:
aggregate = [
  {
    "$match": {
      "data_description.project_name": {
        "$regex": "Allen Brain Observatory - Visual Coding Ophys",
        "$options": "i"
      },
    }
  },
  {
    "$project": {
      "name": 1, 
      "subject_id": "$data_description.subject_id",
      "genotype": "$subject.subject_details.genotype", 
      "date_of_birth": "$subject.subject_details.date_of_birth", 
      "sex": "$subject.subject_details.sex", 
      "session_time": "$acquisition.acquisition_start_time",
      "project_name": "$data_description.project_name", 
      "modality": "$data_description.modalities.name",
      "depth": "$acquisition.data_streams.configurations.images.planes.depth",
      "targeted_structure": "$acquisition.data_streams.configurations.images.planes.targeted_structure.acronym",
      "session_type": "$acquisition.acquisition_type",
      "container_id": { "$arrayElemAt": ["$data_description.tags", 2] },
    }
  },
]
    
records = docdb_api_client.aggregate_docdb_records(
    pipeline = aggregate,
)

In [4]:
df = pd.DataFrame(records)

df['session_date'] = df.apply(lambda x: datetime.fromisoformat(x['session_time']).date(), axis=1)
df['session_time'] = df.apply(lambda x: datetime.fromisoformat(x['session_time']).time(), axis=1)
df['date_of_birth'] = df.apply(lambda x: datetime.strptime(x['date_of_birth'], '%Y-%m-%d').date(), axis=1)
df['age'] = df.apply(lambda x: (x['session_date'] - x['date_of_birth']).days, axis=1)

df['depth'] = df.apply(lambda x: list(np.array(x['depth']).flatten())[0], axis=1)
df['targeted_structure'] = df.apply(lambda x: str(np.array(x['targeted_structure']).flatten()[0]), axis=1)

df['container_id'] = df.apply(lambda x: int(x['container_id'].split(' ')[-1]), axis=1)

order = ['project_name','_id','name','subject_id','genotype','date_of_birth','sex','modality',
         'session_type','session_date','age','session_time','depth', 'targeted_structure','container_id']
df = df[order]

df.head()

,project_name,_id,name,subject_id,genotype,date_of_birth,sex,modality,session_type,session_date,age,session_time,depth,targeted_structure,container_id
0,Allen Brain Observatory - Visual Coding Ophys,c88f22d2-a6a5-42a8-805a-80869635cfba,261967_2016-11-08_11-29-45_nwb_2026-08-19_17-4...,261967,Nr5a1-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-GCaMP6f)/wt,2016-06-13,Female,"[Behavior videos, Planar optical physiology]",three_session_B,2016-11-08,148,11:29:45,350,VISpm,555749366
1,Allen Brain Observatory - Visual Coding Ophys,ae362a40-f97b-495e-96ec-fea98b19d753,280643_2016-12-22_11-09-41_nwb_2026-08-19_17-1...,280643,Emx1-IRES-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-GCaMP...,2016-09-17,Male,"[Planar optical physiology, Behavior videos]",three_session_B,2016-12-22,96,11:09:41,175,VISal,560876149
2,Allen Brain Observatory - Visual Coding Ophys,f1335249-f136-4b7b-a9f2-6b2b920d8c25,229105_2016-03-09_11-13-50_nwb_2026-08-19_18-1...,229105,Cux2-CreERT2/Cux2-CreERT2;Camk2a-tTA/wt;Ai93(T...,2015-12-14,Male,"[Planar optical physiology, Behavior videos]",three_session_C,2016-03-09,86,11:13:50,175,VISal,511510998
3,Allen Brain Observatory - Visual Coding Ophys,e556a3a4-edba-423e-a0eb-f239bf42e6cd,361635_2018-02-19_11-00-16_nwb_2026-08-19_18-1...,361635,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai93(TITL-G...,2017-10-12,Female,"[Planar optical physiology, Behavior videos]",three_session_A,2018-02-19,130,11:00:16,375,VISpm,665413463
4,Allen Brain Observatory - Visual Coding Ophys,8a249c0b-ce8e-4b28-804b-e05755280db2,336246_2017-08-30_11-08-01_nwb_2026-08-19_17-2...,336246,Vip-IRES-Cre/wt;Ai148(TIT2L-GC6f-ICL-tTA2)/wt,2017-06-03,Female,"[Planar optical physiology, Behavior videos]",three_session_A,2017-08-30,88,11:08:01,275,VISl,614556104


In [6]:
df.to_csv('/data/metadata/visual_coding_ophys_metadata.csv', index= False)